# More on the Multivariate Chain Rule

## Introduction

In the previous notebook, we derived the multivariate chain rule:

$$\frac{df}{dt} = \nabla f \cdot \frac{d\mathbf{x}}{dt}$$

In this notebook, we'll explore:

1. **The Jacobian Connection**: How the chain rule relates to the Jacobian matrix we learned earlier
2. **Multi-Link Chains**: Extending the chain rule to more than two functions
3. **Vector-Valued Functions**: Handling cases where intermediate functions are vector-valued

These concepts are crucial for understanding:
- Deep neural networks (many layers of composition)
- Backpropagation (gradient flow through multiple layers)
- Complex computational graphs in machine learning

In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sympy as sp
from matplotlib import cm
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# For better visualization
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Initialize sympy printing
sp.init_printing()

## Part 1: The Jacobian Connection

### Recall: Our Chain Rule Formula

From the previous notebook, we had:

$$\frac{df}{dt} = \frac{\partial f}{\partial \mathbf{x}} \cdot \frac{d\mathbf{x}}{dt}$$

where $\frac{\partial f}{\partial \mathbf{x}}$ is a vector of partial derivatives.

### The Key Insight: This is the Jacobian!

The vector $\frac{\partial f}{\partial \mathbf{x}}$ is exactly the **Jacobian** we learned in the previous module! There's just one small difference:

- **Jacobian (as we learned it)**: A **row vector** $J_f = \begin{bmatrix} \frac{\partial f}{\partial x_1} & \frac{\partial f}{\partial x_2} & \cdots & \frac{\partial f}{\partial x_n} \end{bmatrix}$

- **Gradient vector**: A **column vector** $\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$

**Relationship**: $\frac{\partial f}{\partial \mathbf{x}} = (\nabla f)^T = J_f^T$

The gradient is the transpose of the Jacobian!

### Why Does This Matter?

Using the Jacobian notation, we can rewrite the chain rule:

$$\boxed{\frac{df}{dt} = J_f \cdot \frac{d\mathbf{x}}{dt}}$$

where:
- $J_f$ is a **row vector** (Jacobian of $f$)
- $\frac{d\mathbf{x}}{dt}$ is a **column vector**
- Their product is a **scalar** (as expected for $\frac{df}{dt}$)

This is just **matrix multiplication** of a row vector times a column vector!

In [2]:
# Let's demonstrate with a concrete example
x, y, z, t = sp.symbols('x y z t')

# Define a function f(x, y, z)
f = x**2 + 2*y*z + z**3

print("="*70)
print("DEMONSTRATING THE JACOBIAN REPRESENTATION")
print("="*70)

print(f"\nFunction: f(x, y, z) = {f}")

# Compute gradient (column vector)
gradient = sp.Matrix([
    sp.diff(f, x),
    sp.diff(f, y),
    sp.diff(f, z)
])

print("\n" + "-"*70)
print("Gradient (column vector):")
print("-"*70)
print("∇f =")
sp.pprint(gradient)

# Compute Jacobian (row vector)
jacobian = gradient.T

print("\n" + "-"*70)
print("Jacobian (row vector):")
print("-"*70)
print("J_f =")
sp.pprint(jacobian)

print("\n" + "-"*70)
print("Relationship:")
print("-"*70)
print("J_f = (∇f)^T  (Jacobian is the transpose of the gradient)")

# Now let's use both forms with the chain rule
print("\n" + "="*70)
print("CHAIN RULE WITH BOTH NOTATIONS")
print("="*70)

# Define dx/dt as a column vector
dx_dt = sp.Matrix([sp.Symbol('dx/dt'), sp.Symbol('dy/dt'), sp.Symbol('dz/dt')])

print("\ndx/dt (column vector) =")
sp.pprint(dx_dt)

# Method 1: Dot product (gradient form)
print("\n" + "-"*70)
print("Method 1: Using gradient (dot product)")
print("-"*70)
print("df/dt = ∇f · (dx/dt)")

df_dt_method1 = gradient.dot(dx_dt)
print("\ndf/dt =")
sp.pprint(df_dt_method1)

# Method 2: Matrix multiplication (Jacobian form)
print("\n" + "-"*70)
print("Method 2: Using Jacobian (matrix multiplication)")
print("-"*70)
print("df/dt = J_f × (dx/dt)")

df_dt_method2 = jacobian * dx_dt
print("\ndf/dt =")
sp.pprint(df_dt_method2[0])

print("\n" + "="*70)
print("✓ Both methods give the same result!")
print("="*70)

DEMONSTRATING THE JACOBIAN REPRESENTATION

Function: f(x, y, z) = x**2 + 2*y*z + z**3

----------------------------------------------------------------------
Gradient (column vector):
----------------------------------------------------------------------
∇f =
⎡   2⋅x    ⎤
⎢          ⎥
⎢   2⋅z    ⎥
⎢          ⎥
⎢         2⎥
⎣2⋅y + 3⋅z ⎦

----------------------------------------------------------------------
Jacobian (row vector):
----------------------------------------------------------------------
J_f =
⎡                   2⎤
⎣2⋅x  2⋅z  2⋅y + 3⋅z ⎦

----------------------------------------------------------------------
Relationship:
----------------------------------------------------------------------
J_f = (∇f)^T  (Jacobian is the transpose of the gradient)

CHAIN RULE WITH BOTH NOTATIONS

dx/dt (column vector) =
⎡dx/dt⎤
⎢     ⎥
⎢dy/dt⎥
⎢     ⎥
⎣dz/dt⎦

----------------------------------------------------------------------
Method 1: Using gradient (dot product)
---------------------

## Part 2: Extending the Chain Rule to Multiple Links

### The Question

What if we have **more than two functions** in our chain? Does the chain rule still work?

**Spoiler**: Yes! And it's beautiful. 🎉

### Starting Simple: Univariate Multi-Link Example

Before diving into the multivariate case, let's see how this works with single-variable functions.

#### Example Setup

Consider three functions chained together:

$$f(x) = 5x$$
$$x(u) = 1 - u$$
$$u(t) = t^2$$

So we have: $f(x(u(t)))$ - three functions linking $t$ to $f$.

```
t → u(t) → x(u) → f(x)
```

**Question**: What is $\frac{df}{dt}$?

In [3]:
# Define symbolic variables
t, u, x = sp.symbols('t u x')

# Define the three functions
f_of_x = 5 * x
x_of_u = 1 - u
u_of_t = t**2

print("="*70)
print("UNIVARIATE MULTI-LINK CHAIN RULE")
print("="*70)

print("\nThree functions:")
print(f"  f(x) = {f_of_x}")
print(f"  x(u) = {x_of_u}")
print(f"  u(t) = {u_of_t}")

print("\n" + "="*70)
print("METHOD 1: Direct Substitution")
print("="*70)

# Substitute step by step
x_in_terms_of_u = x_of_u
f_in_terms_of_u = f_of_x.subs(x, x_in_terms_of_u)
print(f"\nStep 1: f in terms of u:")
print(f"  f(u) = 5 × ({x_of_u}) = {f_in_terms_of_u}")

f_in_terms_of_t = f_in_terms_of_u.subs(u, u_of_t)
print(f"\nStep 2: f in terms of t:")
print(f"  f(t) = {f_in_terms_of_u} with u = {u_of_t}")
print(f"  f(t) = {f_in_terms_of_t}")

# Differentiate directly
df_dt_direct = sp.diff(f_in_terms_of_t, t)
print(f"\nStep 3: Differentiate directly:")
print(f"  df/dt = {df_dt_direct}")

print("\n" + "="*70)
print("METHOD 2: Multi-Link Chain Rule")
print("="*70)

print("\nThe multi-link chain rule:")
print("  df/dt = (df/dx) × (dx/du) × (du/dt)")
print("\nThis extends the regular chain rule to three links!")

# Compute each derivative
df_dx = sp.diff(f_of_x, x)
dx_du = sp.diff(x_of_u, u)
du_dt = sp.diff(u_of_t, t)

print(f"\nCompute each piece:")
print(f"  df/dx = {df_dx}")
print(f"  dx/du = {dx_du}")
print(f"  du/dt = {du_dt}")

# Apply chain rule
df_dt_chain = df_dx * dx_du * du_dt
print(f"\nMultiply them together:")
print(f"  df/dt = ({df_dx}) × ({dx_du}) × ({du_dt})")
print(f"  df/dt = {df_dt_chain}")

print("\n" + "="*70)
print("VERIFICATION")
print("="*70)
print(f"Method 1 (direct): df/dt = {df_dt_direct}")
print(f"Method 2 (chain):  df/dt = {df_dt_chain}")
print(f"\n✓ Both methods give the same answer!")

print("\n" + "="*70)
print("KEY INSIGHT")
print("="*70)
print("""
The chain rule extends naturally to any number of links!

For f(x(u(t))):  df/dt = (df/dx) × (dx/du) × (du/dt)

For even more links, just keep multiplying:
  df/dt = (df/dx) × (dx/du) × (du/dv) × (dv/dw) × ... × (dw/dt)

Each intermediate derivative is computed independently,
then they're all multiplied together!
""")

UNIVARIATE MULTI-LINK CHAIN RULE

Three functions:
  f(x) = 5*x
  x(u) = 1 - u
  u(t) = t**2

METHOD 1: Direct Substitution

Step 1: f in terms of u:
  f(u) = 5 × (1 - u) = 5 - 5*u

Step 2: f in terms of t:
  f(t) = 5 - 5*u with u = t**2
  f(t) = 5 - 5*t**2

Step 3: Differentiate directly:
  df/dt = -10*t

METHOD 2: Multi-Link Chain Rule

The multi-link chain rule:
  df/dt = (df/dx) × (dx/du) × (du/dt)

This extends the regular chain rule to three links!

Compute each piece:
  df/dx = 5
  dx/du = -1
  du/dt = 2*t

Multiply them together:
  df/dt = (5) × (-1) × (2*t)
  df/dt = -10*t

VERIFICATION
Method 1 (direct): df/dt = -10*t
Method 2 (chain):  df/dt = -10*t

✓ Both methods give the same answer!

KEY INSIGHT

The chain rule extends naturally to any number of links!

For f(x(u(t))):  df/dt = (df/dx) × (dx/du) × (du/dt)

For even more links, just keep multiplying:
  df/dt = (df/dx) × (dx/du) × (du/dv) × (dv/dw) × ... × (dw/dt)

Each intermediate derivative is computed independently,
th

In [ ]:
# Visualize the multi-link chain
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: The functions as separate plots
ax1 = axes[0]
t_vals = np.linspace(-2, 2, 200)
u_vals = t_vals**2
x_vals = 1 - u_vals
f_vals = 5 * x_vals

ax1.plot(t_vals, u_vals, 'b-', linewidth=2, label='u(t) = t²')
ax1.plot(t_vals, x_vals, 'g-', linewidth=2, label='x(u) = 1 - u')
ax1.plot(t_vals, f_vals, 'r-', linewidth=2, label='f(x) = 5x')
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax1.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('t or u or x (depending on function)', fontsize=12)
ax1.set_ylabel('Output value', fontsize=12)
ax1.set_title('The Three Functions in Our Chain', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: The composition f(x(u(t)))
ax2 = axes[1]
t_vals_2 = np.linspace(-2, 2, 200)
f_of_t_vals = 5 - 5*t_vals_2**2

# Also plot the derivative
df_dt_vals = -10 * t_vals_2

ax2.plot(t_vals_2, f_of_t_vals, 'purple', linewidth=3, label='f(t) = 5 - 5t²')
ax2_twin = ax2.twinx()
ax2_twin.plot(t_vals_2, df_dt_vals, 'orange', linewidth=2, linestyle='--', 
              label="df/dt = -10t", alpha=0.8)

ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('t', fontsize=12)
ax2.set_ylabel('f(t)', fontsize=12, color='purple')
ax2_twin.set_ylabel('df/dt', fontsize=12, color='orange')
ax2.set_title('The Final Composition: f(x(u(t))) and its Derivative', 
              fontsize=14, fontweight='bold')
ax2.legend(loc='upper left', fontsize=11)
ax2_twin.legend(loc='upper right', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='y', labelcolor='purple')
ax2_twin.tick_params(axis='y', labelcolor='orange')

plt.tight_layout()
plt.show()

print("📊 Visual Interpretation:")
print("  • Top plot: The three individual functions")
print("  • Bottom plot: The final composition f(t) (purple) and its derivative df/dt (orange)")
print("  • The chain rule lets us compute df/dt without explicitly finding f(t) first!")

## Part 3: The Multivariate Multi-Link Chain Rule

Now for the main event! What happens when we have **multiple links** AND **vector-valued functions**?

### The Setup

Consider the composition:

$$f(\mathbf{x}(\mathbf{u}(t)))$$

where:
- $f$ is a **scalar-valued** function that takes vector $\mathbf{x}$ as input
- $\mathbf{x}$ is a **vector-valued** function that takes vector $\mathbf{u}$ as input  
- $\mathbf{u}$ is a **vector-valued** function that takes scalar $t$ as input

### The Chain

```
        scalar          vector          vector          scalar
          t      →      u(t)     →      x(u)     →      f(x)
```

We're relating a scalar input $t$ to a scalar output $f$ through two intermediate vector-valued functions!

### The Multi-Link Chain Rule

$$\boxed{\frac{df}{dt} = J_f \cdot J_x \cdot \frac{d\mathbf{u}}{dt}}$$

where:
- $J_f$ = Jacobian of $f$ with respect to $\mathbf{x}$ (a **row vector**)
- $J_x$ = Jacobian of $\mathbf{x}$ with respect to $\mathbf{u}$ (a **matrix**)
- $\frac{d\mathbf{u}}{dt}$ = derivative of $\mathbf{u}$ with respect to $t$ (a **column vector**)

### Understanding the Dimensions

Let's say:
- $\mathbf{u}$ is an $m$-dimensional vector
- $\mathbf{x}$ is an $n$-dimensional vector
- $f$ is a scalar

Then the dimensions work out as:

$$\underbrace{\frac{df}{dt}}_{\text{scalar}} = \underbrace{J_f}_{1 \times n} \cdot \underbrace{J_x}_{n \times m} \cdot \underbrace{\frac{d\mathbf{u}}{dt}}_{m \times 1}$$

$$\text{Result: } (1 \times n) \cdot (n \times m) \cdot (m \times 1) = 1 \times 1 = \text{scalar} \,\checkmark$$

The dimensions match perfectly for matrix multiplication!

### Understanding the Middle Term: $J_x$ (The Jacobian Matrix)

The middle term $J_x = \frac{\partial \mathbf{x}}{\partial \mathbf{u}}$ is the most interesting!

When $\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$ depends on $\mathbf{u} = \begin{bmatrix} u_1 \\ u_2 \end{bmatrix}$, we need:

**The derivative of each output variable with respect to each input variable.**

This gives us **four terms** arranged in a matrix:

$$J_x = \frac{\partial \mathbf{x}}{\partial \mathbf{u}} = \begin{bmatrix}
\frac{\partial x_1}{\partial u_1} & \frac{\partial x_1}{\partial u_2} \\
\frac{\partial x_2}{\partial u_1} & \frac{\partial x_2}{\partial u_2}
\end{bmatrix}$$

This is the **Jacobian matrix** we learned in the previous module!

### Summary of Terms

| Term | Name | Dimensions | What it represents |
|------|------|------------|-------------------|
| $J_f$ | Jacobian of $f$ | $1 \times n$ (row) | How $f$ changes with each component of $\mathbf{x}$ |
| $J_x$ | Jacobian of $\mathbf{x}$ | $n \times m$ (matrix) | How each component of $\mathbf{x}$ changes with each component of $\mathbf{u}$ |
| $\frac{d\mathbf{u}}{dt}$ | Derivative of $\mathbf{u}$ | $m \times 1$ (column) | How each component of $\mathbf{u}$ changes with $t$ |

## Example: Concrete Multi-Link Multivariate Chain Rule

Let's work through a complete example with 2D vectors.

### Problem Setup

Given:

**Function $f$:** (scalar-valued)
$$f(x_1, x_2) = x_1^2 + x_1 x_2$$

**Function $\mathbf{x}$:** (vector-valued, takes $\mathbf{u}$ as input)
$$\mathbf{x}(\mathbf{u}) = \begin{bmatrix} x_1(u_1, u_2) \\ x_2(u_1, u_2) \end{bmatrix} = \begin{bmatrix} u_1 + u_2 \\ u_1 - u_2 \end{bmatrix}$$

**Function $\mathbf{u}$:** (vector-valued, takes $t$ as input)
$$\mathbf{u}(t) = \begin{bmatrix} u_1(t) \\ u_2(t) \end{bmatrix} = \begin{bmatrix} t^2 \\ 2t \end{bmatrix}$$

**Find:** $\frac{df}{dt}$ using the multi-link chain rule

In [ ]:
# Define symbolic variables
t = sp.Symbol('t')
x1, x2 = sp.symbols('x1 x2')
u1, u2 = sp.symbols('u1 u2')

print("="*80)
print("MULTIVARIATE MULTI-LINK CHAIN RULE EXAMPLE")
print("="*80)

# Define the functions
f = x1**2 + x1*x2
x1_func = u1 + u2
x2_func = u1 - u2
u1_func = t**2
u2_func = 2*t

print("\nGiven:")
print(f"  f(x₁, x₂) = {f}")
print(f"  x₁(u₁, u₂) = {x1_func}")
print(f"  x₂(u₁, u₂) = {x2_func}")
print(f"  u₁(t) = {u1_func}")
print(f"  u₂(t) = {u2_func}")

print("\n" + "="*80)
print("STEP 1: Compute J_f (Jacobian of f with respect to x)")
print("="*80)
print("This is a 1×2 row vector")

df_dx1 = sp.diff(f, x1)
df_dx2 = sp.diff(f, x2)

J_f = sp.Matrix([[df_dx1, df_dx2]])
print(f"\n∂f/∂x₁ = {df_dx1}")
print(f"∂f/∂x₂ = {df_dx2}")
print(f"\nJ_f = [∂f/∂x₁  ∂f/∂x₂] =")
sp.pprint(J_f)

print("\n" + "="*80)
print("STEP 2: Compute J_x (Jacobian of x with respect to u)")
print("="*80)
print("This is a 2×2 matrix")

dx1_du1 = sp.diff(x1_func, u1)
dx1_du2 = sp.diff(x1_func, u2)
dx2_du1 = sp.diff(x2_func, u1)
dx2_du2 = sp.diff(x2_func, u2)

J_x = sp.Matrix([
    [dx1_du1, dx1_du2],
    [dx2_du1, dx2_du2]
])

print(f"\n∂x₁/∂u₁ = {dx1_du1}    ∂x₁/∂u₂ = {dx1_du2}")
print(f"∂x₂/∂u₁ = {dx2_du1}    ∂x₂/∂u₂ = {dx2_du2}")
print(f"\nJ_x =")
sp.pprint(J_x)

print("\n" + "="*80)
print("STEP 3: Compute du/dt (derivative of u with respect to t)")
print("="*80)
print("This is a 2×1 column vector")

du1_dt = sp.diff(u1_func, t)
du2_dt = sp.diff(u2_func, t)

du_dt = sp.Matrix([du1_dt, du2_dt])

print(f"\ndu₁/dt = {du1_dt}")
print(f"du₂/dt = {du2_dt}")
print(f"\ndu/dt =")
sp.pprint(du_dt)

print("\n" + "="*80)
print("STEP 4: Apply the chain rule: df/dt = J_f × J_x × (du/dt)")
print("="*80)
print("\nDimension check:")
print(f"  J_f:    {J_f.shape[0]}×{J_f.shape[1]} (row vector)")
print(f"  J_x:    {J_x.shape[0]}×{J_x.shape[1]} (matrix)")
print(f"  du/dt:  {du_dt.shape[0]}×{du_dt.shape[1]} (column vector)")
print(f"  Result: (1×2) × (2×2) × (2×1) = 1×1 (scalar) ✓")

# Compute the chain rule
result = J_f * J_x * du_dt
df_dt_symbolic = result[0, 0]

print(f"\ndf/dt = J_f × J_x × (du/dt)")
print(f"      = {df_dt_symbolic}")

print("\n" + "="*80)
print("STEP 5: Substitute back the actual functions")
print("="*80)

# Substitute u values
df_dt_with_u = df_dt_symbolic.subs({u1: u1_func, u2: u2_func})
print(f"\nSubstitute u₁ = {u1_func}, u₂ = {u2_func}:")
print(f"df/dt = {df_dt_with_u}")

# Expand
df_dt_expanded = sp.expand(df_dt_with_u)
print(f"\nExpanded:")
print(f"df/dt = {df_dt_expanded}")

# Factor if possible
df_dt_factored = sp.factor(df_dt_expanded)
print(f"\nFactored:")
print(f"df/dt = {df_dt_factored}")

print("\n" + "="*80)
print("✓ SUCCESS! We've computed df/dt using the multi-link chain rule!")
print("="*80)

### Visual Summary of the Computation

Let's visualize the dimensions and flow of computation:

```
                DIMENSION FLOW

                    t
                    ↓
              [u₁, u₂]     ← 2×1 vector
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
      [x₁, x₂]     ← 2×1 vector
          ↓
          f        ← scalar


                CHAIN RULE COMPUTATION

    df/dt = J_f      ×      J_x      ×    du/dt
            ↓                ↓              ↓
          [1×2]           [2×2]          [2×1]
            ↓                ↓              ↓
       [∂f/∂x₁ ∂f/∂x₂]  [∂x₁/∂u₁  ∂x₁/∂u₂]  [du₁/dt]
                        [∂x₂/∂u₁  ∂x₂/∂u₂]  [du₂/dt]
            ↓                ↓              ↓
          ───────────────────────────────────
                        ↓
                    [1×1] = scalar
```

**Key Insight**: The Jacobian matrices handle all the partial derivatives systematically, and matrix multiplication combines them correctly!

### Verification: Does it match direct substitution?

Let's verify our answer by directly substituting everything and differentiating.

In [ ]:
# Verification by direct substitution
print("="*80)
print("VERIFICATION: Direct Substitution Method")
print("="*80)

# Start with f(x₁, x₂)
f_original = x1**2 + x1*x2

# Substitute x₁ = u₁ + u₂, x₂ = u₁ - u₂
f_in_u = f_original.subs({x1: u1 + u2, x2: u1 - u2})
print(f"\nStep 1: Substitute x in terms of u")
print(f"f(u₁, u₂) = {f_in_u}")
f_in_u_expanded = sp.expand(f_in_u)
print(f"Expanded: {f_in_u_expanded}")

# Substitute u₁ = t², u₂ = 2t
f_in_t = f_in_u_expanded.subs({u1: t**2, u2: 2*t})
print(f"\nStep 2: Substitute u in terms of t")
print(f"f(t) = {f_in_t}")
f_in_t_expanded = sp.expand(f_in_t)
print(f"Expanded: {f_in_t_expanded}")

# Differentiate directly
df_dt_direct = sp.diff(f_in_t_expanded, t)
print(f"\nStep 3: Differentiate with respect to t")
print(f"df/dt = {df_dt_direct}")

# Factor for comparison
df_dt_direct_factored = sp.factor(df_dt_direct)
print(f"Factored: {df_dt_direct_factored}")

print("\n" + "="*80)
print("COMPARISON")
print("="*80)
print(f"\nChain rule method: df/dt = {df_dt_factored}")
print(f"Direct method:     df/dt = {df_dt_direct_factored}")

# Check if they're equal
difference = sp.simplify(df_dt_factored - df_dt_direct_factored)
print(f"\nDifference: {difference}")

if difference == 0:
    print("\n✓✓✓ BOTH METHODS GIVE THE SAME ANSWER! ✓✓✓")
else:
    print("\n⚠ Methods give different answers (check calculations)")

print("\n" + "="*80)
print("KEY TAKEAWAY")
print("="*80)
print("""
The multi-link chain rule works perfectly!

Instead of doing messy substitutions, we:
1. Compute each Jacobian independently
2. Multiply them together using matrix multiplication
3. Get the same answer, but with much clearer structure!

This modular approach is essential for:
• Deep learning (many layers)
• Automatic differentiation
• Complex computational graphs
""")

## The Beautiful Connection: Linear Algebra Meets Calculus

### What We've Discovered

The transcript mentions: *"I hope you're now starting to see the various threads of linear algebra and multivariate calculus weaved together."*

Let's make this connection explicit!

### The Weaving of Concepts

| **Linear Algebra** | **Meets** | **Calculus** | **Result** |
|-------------------|-----------|--------------|------------|
| Row vectors | × | Partial derivatives | = Jacobian (gradient) |
| Matrices | × | Multivariate functions | = Jacobian matrix |
| Matrix multiplication | × | Chain rule | = Composition of derivatives |
| Vector spaces | × | Function spaces | = Transformations |

### Why This Matters

1. **Jacobians as Linear Transformations**
   - Each Jacobian represents a local linear approximation
   - Matrix multiplication composes these transformations
   
2. **Dimension Compatibility**
   - Matrix dimensions must match for multiplication
   - This automatically ensures the chain rule is applied correctly
   
3. **Computational Efficiency**
   - Matrix operations are highly optimized in computers
   - Linear algebra libraries (NumPy, PyTorch, TensorFlow) handle this automatically

4. **Backpropagation in Neural Networks**
   - Each layer has a Jacobian
   - Backprop multiplies Jacobians from output to input
   - This IS the multi-link chain rule in action!

### The General Pattern

For any composition of functions:

$$f(\mathbf{x}_n(\mathbf{x}_{n-1}(\ldots \mathbf{x}_2(\mathbf{x}_1(t))\ldots)))$$

The derivative is:

$$\frac{df}{dt} = J_f \cdot J_{x_n} \cdot J_{x_{n-1}} \cdot \ldots \cdot J_{x_2} \cdot J_{x_1} \cdot \frac{dt}{dt}$$

Just keep multiplying Jacobians! 🎯

In [ ]:
# Visualize dimension compatibility in the chain rule
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Title
ax.text(0.5, 0.95, 'Multi-Link Chain Rule: Dimension Flow', 
        ha='center', fontsize=18, fontweight='bold', transform=ax.transAxes)

# Layer 1: The chain
y_start = 0.85
x_positions = [0.15, 0.35, 0.55, 0.75]
layer_names = ['t\n(scalar)', 'u\n(vector)', 'x\n(vector)', 'f\n(scalar)']
layer_dims = ['1', 'm', 'n', '1']

for i, (x, name, dim) in enumerate(zip(x_positions, layer_names, layer_dims)):
    # Draw box
    box = FancyBboxPatch((x - 0.05, y_start - 0.05), 0.1, 0.1,
                         boxstyle="round,pad=0.01", 
                         edgecolor='blue', facecolor='lightblue',
                         linewidth=2, transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x, y_start, name, ha='center', va='center', fontsize=12,
           transform=ax.transAxes, fontweight='bold')
    ax.text(x, y_start - 0.08, f'dim: {dim}', ha='center', va='top', 
           fontsize=9, transform=ax.transAxes, style='italic')
    
    # Draw arrow
    if i < len(x_positions) - 1:
        ax.annotate('', xy=(x_positions[i+1] - 0.05, y_start), 
                   xytext=(x + 0.05, y_start),
                   arrowprops=dict(arrowstyle='->', lw=2, color='black'),
                   transform=ax.transAxes)

# Layer 2: The derivatives
y_deriv = 0.65
derivative_labels = ['du/dt', 'J_x = ∂x/∂u', 'J_f = ∂f/∂x']
derivative_dims = ['m × 1', 'n × m', '1 × n']
x_deriv_positions = [0.25, 0.45, 0.65]

for x, label, dim in zip(x_deriv_positions, derivative_labels, derivative_dims):
    # Draw box for derivative
    box = FancyBboxPatch((x - 0.07, y_deriv - 0.04), 0.14, 0.08,
                         boxstyle="round,pad=0.01",
                         edgecolor='darkgreen', facecolor='lightgreen',
                         linewidth=2, transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x, y_deriv, label, ha='center', va='center', fontsize=11,
           transform=ax.transAxes)
    ax.text(x, y_deriv - 0.07, f'dim: {dim}', ha='center', va='top',
           fontsize=9, transform=ax.transAxes, style='italic', color='darkgreen')

# Layer 3: The multiplication
y_mult = 0.45
ax.text(0.45, y_mult, 'df/dt = J_f × J_x × du/dt', 
       ha='center', va='center', fontsize=14, fontweight='bold',
       transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7, edgecolor='orange', linewidth=2))

# Layer 4: Dimension calculation
y_calc = 0.32
dim_calc = '(1 × n) × (n × m) × (m × 1) = 1 × 1'
ax.text(0.45, y_calc, dim_calc, 
       ha='center', va='center', fontsize=13, 
       transform=ax.transAxes, family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.7, edgecolor='blue', linewidth=1))

# Add checkmark
ax.text(0.45, y_calc - 0.08, '✓ Dimensions match!', 
       ha='center', va='center', fontsize=12, color='green',
       transform=ax.transAxes, fontweight='bold')

# Bottom explanation
explanation = """
Key Insight: Matrix dimensions automatically ensure correct composition!
The 'inner dimensions' must match: (1×n) × (n×m) × (m×1)
The result has the 'outer dimensions': 1×1 (a scalar)
"""
ax.text(0.45, 0.08, explanation,
       ha='center', va='center', fontsize=10,
       transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5, edgecolor='brown'))

plt.tight_layout()
plt.show()

print("\n📊 This diagram shows how:")
print("  • Each function transforms the dimensions")
print("  • Each Jacobian has dimensions that match for multiplication")
print("  • The final result is a scalar (1×1), as expected for df/dt")

## Key Takeaways

### 1. The Jacobian Representation

The chain rule from the previous notebook:
$$\frac{df}{dt} = \nabla f \cdot \frac{d\mathbf{x}}{dt}$$

Can be written using Jacobians:
$$\boxed{\frac{df}{dt} = J_f \cdot \frac{d\mathbf{x}}{dt}}$$

where $J_f$ is a **row vector** (transpose of the gradient).

### 2. Multi-Link Chains Work!

The chain rule extends to any number of intermediate functions:

**Univariate**: $\frac{df}{dt} = \frac{df}{dx} \cdot \frac{dx}{du} \cdot \frac{du}{dt}$

**Multivariate**: $\frac{df}{dt} = J_f \cdot J_x \cdot \frac{d\mathbf{u}}{dt}$

**Many links**: $\frac{df}{dt} = J_f \cdot J_{x_n} \cdot J_{x_{n-1}} \cdot \ldots \cdot J_{x_1} \cdot \frac{dt}{dt}$

### 3. The Middle Term is a Matrix

When intermediate functions are vector-valued:
- $J_f$ is a **row vector** (1 × n)
- $J_x$ is a **matrix** (n × m)  ← This is new!
- $\frac{d\mathbf{u}}{dt}$ is a **column vector** (m × 1)

The Jacobian matrix $J_x$ contains all partial derivatives of each output with respect to each input.

### 4. Dimensions Must Match

Matrix multiplication requires dimension compatibility:
$$(1 \times n) \cdot (n \times m) \cdot (m \times 1) = (1 \times 1)$$

The "inner dimensions" must match, and the result has the "outer dimensions".

### 5. This IS Backpropagation!

In neural networks:
- Each layer is a function with a Jacobian
- Backprop multiplies Jacobians from output to input
- This is exactly the multi-link multivariate chain rule
- The chain can be arbitrarily long (deep networks)

## Summary: The Complete Picture

### What We Learned

Starting from the simple chain rule, we've built up to a complete understanding:

#### Level 1: Basic Chain Rule (Single Variable)
$$\frac{df}{dt} = \frac{df}{dx} \cdot \frac{dx}{dt}$$

#### Level 2: Multivariate Chain Rule (Vectors)
$$\frac{df}{dt} = J_f \cdot \frac{d\mathbf{x}}{dt}$$

#### Level 3: Multi-Link Univariate
$$\frac{df}{dt} = \frac{df}{dx} \cdot \frac{dx}{du} \cdot \frac{du}{dt}$$

#### Level 4: Multi-Link Multivariate (The Complete Form)
$$\boxed{\frac{df}{dt} = J_f \cdot J_x \cdot \frac{d\mathbf{u}}{dt}}$$

where:
- $J_f$ can be a row vector or matrix (depending on output dimension)
- $J_x$ is typically a matrix (vector → vector transformation)
- $\frac{d\mathbf{u}}{dt}$ is a column vector

### The Power of This Framework

1. **Modularity**: Compute each Jacobian independently
2. **Composability**: Chain together any number of functions
3. **Efficiency**: Use optimized linear algebra operations
4. **Generality**: Works for functions of any dimension
5. **Practical**: This is how automatic differentiation works!

### The Weaving of Mathematics

As the transcript beautifully states:

> *"I hope you're now starting to see the various threads of linear algebra and multivariate calculus weaved together."*

We've seen:
- **Calculus** gives us derivatives and rates of change
- **Linear Algebra** gives us matrices and vectors
- **The Jacobian** connects them through the chain rule
- **Matrix multiplication** handles composition automatically

This elegant framework underpins modern machine learning, optimization, and scientific computing!

### Looking Forward

With this foundation, you're now equipped to:
- Understand deep neural networks
- Implement backpropagation
- Work with automatic differentiation libraries
- Optimize complex multivariate functions
- Build computational graphs for any application

**Congratulations!** You've mastered one of the most important concepts in applied mathematics. 🎉